In [2]:
import pandas as pd
import numpy as np
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score     
from sklearn.metrics import roc_auc_score

In [3]:
import sys
sys.executable


'c:\\Users\\yhe\\Desktop\\PHD\\Research\\25spring_Agentic_AI_project\\venv\\Scripts\\python.exe'

In [4]:
import os
print(os.getcwd())

c:\Users\yhe\Desktop\PHD\Research\25spring_Agentic_AI_project\data


In [5]:
diabetes_data = pd.read_csv("../data/diabetes_dataset/diabetes.csv")
diabetes_data

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1


Select numerical features

In [6]:
numeric_features = diabetes_data.select_dtypes(include=['number']).columns.tolist()
numeric_features

['Pregnancies',
 'Glucose',
 'BloodPressure',
 'SkinThickness',
 'Insulin',
 'BMI',
 'DiabetesPedigreeFunction',
 'Age',
 'Outcome']

In [7]:
categorical_features = [col for col in diabetes_data.columns if col not in numeric_features]

In [8]:

diabetes_train, diabetes_test = train_test_split(diabetes_data, test_size=0.2, random_state=42)
diabetes_train.to_parquet("../data/diabetes_dataset/train_cleaned.parquet")
diabetes_test.to_parquet("../data/diabetes_dataset/test_cleaned.parquet")


In [9]:
diabetes_x_train = diabetes_train.drop(['Outcome'], axis=1)
diabetes_x_test = diabetes_test.drop(['Outcome'], axis=1)

diabetes_y_train = diabetes_train['Outcome'] 
diabetes_y_test=diabetes_test['Outcome'] 

In [10]:
diabetes_model=RandomForestClassifier(random_state=42)
diabetes_model.fit(diabetes_x_train, diabetes_y_train)
pre_trained_predictions=diabetes_model.predict(diabetes_x_test)
pre_trained_accuracy = accuracy_score(diabetes_y_test, pre_trained_predictions)
print("Pre-Trained Accuracy:", pre_trained_accuracy)
auc=roc_auc_score(diabetes_y_test, diabetes_model.predict_proba(diabetes_x_test)[:, 1])
print(auc)

Pre-Trained Accuracy: 0.7207792207792207
0.8120293847566575


In [29]:
with open('../data/diabetes_dataset/RF.pkl', 'wb') as f:
    pickle.dump(diabetes_model, f)

In [31]:
feature_desc = [
    'Number of times pregnant.',
'Plasma glucose concentration a 2 hours in an oral glucose tolerance test.',
'Diastolic blood pressure (mm Hg).',
'Triceps skin fold thickness (mm).',
'2-Hour serum insulin (mu U/ml).',
'Body mass index (weight in kg/(height in m)^2).',
'DiabetesPedigreeFunction: Diabetes pedigree function.',
'Age (years).',
]


feature_desc_df = pd.DataFrame({
    "feature_name": list(diabetes_x_test.columns),
    "feature_average": diabetes_x_train.mean().to_list() ,
    "feature_desc": feature_desc,
})
     

dataset_description="This dataset is originally from the National Institute of Diabetes and Digestive and Kidney Diseases. The objective is to predict based on diagnostic measurements whether a patient has diabetes."
target_description="The target is to predict based on diagnostic measurements whether a patient has diabetes (1) or not (0)."
task_description="Predict whether a patient has diabetes (1) or not (0)."

dataset_info={
 "dataset_description": dataset_description,
 "target_description": target_description,
 "task_description": task_description,
 "feature_description": feature_desc_df
 }


with open('../data/diabetes_dataset/dataset_info', 'wb') as f:
    pickle.dump(dataset_info, f)

In [32]:
feature_desc_df

,feature_name,feature_average,feature_desc
0,Pregnancies,3.742671,Number of times pregnant.
1,Glucose,120.855049,Plasma glucose concentration a 2 hours in an o...
2,BloodPressure,69.415309,Diastolic blood pressure (mm Hg).
3,SkinThickness,20.399023,Triceps skin fold thickness (mm).
4,Insulin,81.438111,2-Hour serum insulin (mu U/ml).
5,BMI,31.983388,Body mass index (weight in kg/(height in m)^2).
6,DiabetesPedigreeFunction,0.469168,DiabetesPedigreeFunction: Diabetes pedigree fu...
7,Age,32.907166,Age (years).
